In [1]:
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Embedding
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.models import Sequential
import numpy as np


sent = [
    'the glass of milk',
    'the glass of juice',
    'the cup of tea',
    'understanding the meaning of words',
    'learning llms'
]

# Define the vocabulary size
voc_size = 10000

# One hot representation for every word

one_hot_rep = [one_hot(words, voc_size) for words in sent]

sent_length = 8
embedded_docs = pad_sequences(one_hot_rep, padding='pre', maxlen=sent_length)
print(embedded_docs)

# feature representation
dim = 10

# Model
model = Sequential()
model.add(Embedding(voc_size, dim, input_length=sent_length))
model.compile('adam', 'mse')


model.predict(embedded_docs)





[[   0    0    0    0 4489 8658 2538 4971]
 [   0    0    0    0 4489 8658 2538 1240]
 [   0    0    0    0 4489 1671 2538 4964]
 [   0    0    0 6601 4489 9536 2538  494]
 [   0    0    0    0    0    0 3908 9937]]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step


array([[[ 0.03799976,  0.0212608 ,  0.04863698, -0.04053533,
          0.01215632,  0.02586255, -0.01548933, -0.03165028,
         -0.04120452,  0.02711035],
        [ 0.03799976,  0.0212608 ,  0.04863698, -0.04053533,
          0.01215632,  0.02586255, -0.01548933, -0.03165028,
         -0.04120452,  0.02711035],
        [ 0.03799976,  0.0212608 ,  0.04863698, -0.04053533,
          0.01215632,  0.02586255, -0.01548933, -0.03165028,
         -0.04120452,  0.02711035],
        [ 0.03799976,  0.0212608 ,  0.04863698, -0.04053533,
          0.01215632,  0.02586255, -0.01548933, -0.03165028,
         -0.04120452,  0.02711035],
        [-0.00098028, -0.04404525,  0.029109  , -0.03514175,
         -0.0211217 ,  0.04042918,  0.01688294, -0.00304359,
          0.04235956, -0.0201308 ],
        [-0.0153053 ,  0.01860989, -0.03824632, -0.03036115,
         -0.02543389, -0.01431711,  0.03984705,  0.01455419,
         -0.02502508, -0.03764503],
        [-0.04028238,  0.01517278,  0.04553536, -0.0

## Training Simple RNN on IMDB Dataset & Feature Engineering

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense


In [3]:
## Load the imdb dataset

max_features = 10000 # vocab size
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_features)

print(X_train.shape)
print(X_test.shape)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(25000,)
(25000,)


In [4]:
# Inspect a sample review and its label
sample_review = X_train[0]
sample_label = y_train[0]
print(sample_review)
print(sample_label)

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
1


In [8]:
# Mapping of words index back to words (for our understanding)
word_index = imdb.get_word_index()
word_index

#word_index
reverse_word_index = {value: key for key, value in word_index.items()}
reverse_word_index


decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in sample_review])
decoded_review


"? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you th

In [9]:
max_len = 500
X_train = sequence.pad_sequences(X_train, maxlen=max_len)
X_test = sequence.pad_sequences(X_test, maxlen=max_len)
X_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]], dtype=int32)

In [17]:
## Training simple RNN
model=Sequential()
model.add(Embedding(max_features,128,input_length=max_len)) ## Embedding Layers
model.add(SimpleRNN(128,activation='relu'))
model.add(Dense(1,activation="sigmoid"))
model.summary()

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
## Create an instance of EarlyStoppping Callback
from tensorflow.keras.callbacks import EarlyStopping
earlystopping=EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True)
earlystopping

In [ ]:
## Train the model with early sstopping
history=model.fit(
    X_train,y_train,epochs=4,batch_size=32,
    validation_split=0.2,
    callbacks=[earlystopping]
)

Epoch 1/4
625/625 ━━━━━━━━━━━━━━━━━━━━ 143s 228ms/step - accuracy: 0.6668 - loss: 0.6668 - val_accuracy: 0.6790 - val_loss: 0.5854
Epoch 2/4
625/625 ━━━━━━━━━━━━━━━━━━━━ 200s 226ms/step - accuracy: 0.7451 - loss: 114197014773760.0000 - val_accuracy: 0.6088 - val_loss: 0.6550
Epoch 3/4
404/625 ━━━━━━━━━━━━━━━━━━━━ 46s 208ms/step - accuracy: 0.7315 - loss: 0.5293

In [ ]:
## Save model file
model.save('simple_rnn_imdb.h5')